# Trump Tweet Sentiment Classification

This notebook completes the text-cleaning helper function and builds a machine-learning text classification pipeline using TF-IDF and Logistic Regression.

In [20]:
# Optional: run this in Google Colab if your CSV is stored in Google Drive.
# from google.colab import drive
# drive.mount('/content/drive')

## 1. Imports and NLTK setup

In [21]:
import re
import string
from pathlib import Path

import pandas as pd

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split

# Download required NLTK resources.
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

## 2. Helper function for text cleaning

In [22]:
# Create reusable NLP tools once so the function can run faster.
stop_words = set(stopwords.words("english"))

# Optional but useful for sentiment tasks: keep negation words because
# "not good" and "good" should not mean the same thing.
negation_words = {"no", "nor", "not"}
stop_words = stop_words - negation_words

lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()


def remove_emojis(text):
    """Remove common emoji/unicode symbol ranges."""
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"  # emoticons
        "\U0001F300-\U0001F5FF"  # symbols and pictographs
        "\U0001F680-\U0001F6FF"  # transport and map symbols
        "\U0001F1E0-\U0001F1FF"  # flags
        "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251"
        "]+",
        flags=re.UNICODE,
    )
    return emoji_pattern.sub("", text)


def text_cleaning_pipeline(dataset, rule="lemmatize"):
    """
    Clean one text document.

    Parameters
    ----------
    dataset : str
        Raw input text.
    rule : str
        "lemmatize", "stem", or anything else for no stemming/lemmatization.

    Returns
    -------
    str
        Cleaned text.
    """
    # Convert the input to string and lowercase it.
    data = str(dataset).lower()

    # Remove URLs.
    data = re.sub(r"http\S+|www\S+|https\S+", " ", data)

    # Remove Twitter mentions and hashtags symbols.
    # The hashtag word is kept after removing "#".
    data = re.sub(r"@\w+", " ", data)
    data = data.replace("#", "")

    # Remove emojis.
    data = remove_emojis(data)

    # Remove punctuation, digits, and other unwanted characters.
    data = re.sub(r"[^a-zA-Z\s]", " ", data)

    # Remove extra spaces.
    data = re.sub(r"\s+", " ", data).strip()

    # Create tokens.
    tokens = data.split()

    # Remove stopwords and very short tokens.
    tokens = [word for word in tokens if word not in stop_words and len(word) > 1]

    if rule == "lemmatize":
        tokens = [lemmatizer.lemmatize(word) for word in tokens]
    elif rule == "stem":
        tokens = [stemmer.stem(word) for word in tokens]
    elif rule in ["none", None, ""]:
        pass
    else:
        raise ValueError("Pick between 'lemmatize', 'stem', or 'none'.")

    return " ".join(tokens)

In [23]:
# Quick test of the cleaning function
sample_text = "I am NOT happy with this! Visit https://example.com @user #Trump2024 😊"
text_cleaning_pipeline(sample_text, rule="lemmatize")

'not happy visit trump'

## 3. Load the dataset

Place `trump_tweet_sentiment_analysis.csv` in the same folder as this notebook, upload it to `/content/`, or change `DATA_PATH` to the correct Google Drive path.

In [24]:
from google.colab import drive
from pathlib import Path
import pandas as pd

# Mount Google Drive
drive.mount("/content/drive")

# Set the CSV path
DATA_PATH = Path("/content/drive/MyDrive/AI/trum_tweet_sentiment_analysis.csv")

# Read the CSV
df = pd.read_csv(DATA_PATH)

df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,text,Sentiment
0,RT @JohnLeguizamo: #trump not draining swamp b...,0
1,ICYMI: Hackers Rig FM Radio Stations To Play A...,0
2,Trump protests: LGBTQ rally in New York https:...,1
3,"""Hi I'm Piers Morgan. David Beckham is awful b...",0
4,RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...,0


In [25]:
df.columns

Index(['text', 'Sentiment'], dtype='object')

In [26]:
# Rename Sentiment column to label
df = df.rename(columns={"Sentiment": "label"})

# Check that the required columns are present
required_columns = {"text", "label"}
missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# Keep only text and label columns
df = df[["text", "label"]].dropna()
df["text"] = df["text"].astype(str)

print(df.shape)
df["label"].value_counts()

(1850123, 2)


,count
label,
0,1244211
1,605912


## 4. Clean and tokenize the text

In [27]:
df["clean_text"] = df["text"].apply(lambda text: text_cleaning_pipeline(text, rule="lemmatize"))

# Remove rows that become empty after cleaning.
df = df[df["clean_text"].str.strip() != ""].copy()

df[["text", "clean_text", "label"]].head()

,text,clean_text,label
0,RT @JohnLeguizamo: #trump not draining swamp b...,rt trump not draining swamp taxpayer dollar tr...,0
1,ICYMI: Hackers Rig FM Radio Stations To Play A...,icymi hacker rig fm radio station play anti tr...,0
2,Trump protests: LGBTQ rally in New York https:...,trump protest lgbtq rally new york bbcworld via,1
3,"""Hi I'm Piers Morgan. David Beckham is awful b...",hi pier morgan david beckham awful donald trum...,0
4,RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...,rt tech firm suing buzzfeed publishing unverif...,0


## 5. Train-test split

In [28]:
X = df["clean_text"]
y = df["label"]

# Stratify keeps the label proportions similar in train and test sets.
stratify_target = y if y.nunique() > 1 else None

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=stratify_target,
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 1480073
Testing samples: 370019


## 6. TF-IDF vectorization

In [29]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("TF-IDF train shape:", X_train_tfidf.shape)
print("TF-IDF test shape:", X_test_tfidf.shape)

TF-IDF train shape: (1480073, 5000)
TF-IDF test shape: (370019, 5000)


## 7. Model training and evaluation

In [30]:
model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42,
)

model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.8950594428934731

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.90      0.92    248836
           1       0.81      0.89      0.85    121183

    accuracy                           0.90    370019
   macro avg       0.88      0.89      0.88    370019
weighted avg       0.90      0.90      0.90    370019



In [31]:
# Optional: confusion matrix
cm = confusion_matrix(y_test, y_pred)
cm

array([[223713,  25123],
       [ 13707, 107476]])

## 8. Predict sentiment for new text

In [32]:
def predict_sentiment(text):
    cleaned_text = text_cleaning_pipeline(text, rule="lemmatize")
    text_tfidf = vectorizer.transform([cleaned_text])
    prediction = model.predict(text_tfidf)[0]
    return prediction


predict_sentiment("This policy is terrible and I do not like it.")

np.int64(0)